# 🛒 Product Category Classification - Machine Learning Pipeline
- **Author:** Gligor Milićević
- **Project Goal:** Automatically classify product categories based on their titles and metadata using NLP and supervised machine learning models.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import pickle

print("Biblioteke su uspešno učitane!")

Biblioteke su uspešno učitane!


## 📂 1. Data Ingestion & Overview
- Loading the structured product dataset (`product.csv`).
- Inspecting the initial rows, column names, and dataset dimensions.

In [3]:
# Creating the dataset based on the assignment task requirements
data = {
    'product ID': [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21],
    'Product Title': [
        'apple iphone 8 plus 64gb silver', 'apple iphone 8 plus 64 gb spacegrau',
        'apple mq8n2b/a iphone 8 plus 64gb 5.5 12mp sim free smartphone in gold',
        'apple iphone 8 plus 64gb space grey', 'apple iphone 8 plus gold 5.5 64gb 4g unlocked sim free',
        'apple iphone 8 plus gold 5.5 64gb 4g unlocked sim free', 'apple iphone 8 plus 64 gb space grey',
        'apple iphone 8 plus 64gb space grey', 'apple iphone 8 plus 64gb space grey',
        'apple iphone 8 plus 64gb space grey', 'apple iphone 8 plus 5.5 single sim 4g 64gb silver',
        'sim free iphone 8 plus 64gb by apple space grey', 'apple iphone 8 plus 64gb gold smartphone',
        'apple iphone 8 plus 5.5 single sim 4g 64gb grey', 'apple iphone 8 plus silver 5.5 64gb 4g unlocked sim free',
        'apple iphone 8 plus 64 gb silver', 'apple iphone 8 plus 64gb silver unlocked',
        'apple iphone 8 plus 14 cm 5.5 64 gb 12 mp ios 11 silver', 'iphone 8 plus sim free water dust resistant 64gb silver by apple',
        'iphone 8 plus 64gb space grey', 'apple iphone 8 plus'
    ],
    'Merchant ID': [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21],
    'Category Label': ['Mobile Phones']*21,
    '_Product Code': ['QA-2276-XC','KA-2501-QO','FP-8086-IE','YI-0086-US','NZ-3586-WP','SB-2202-CH','QJ-3229-QP','ES-1396-OR','VU-2032-FY','WI-6394-GG','OG-5832-ZN','CB-1674-BJ','ZN-9807-RN','GQ-1449-WG','NE-9826-PD','TO-7503-AY','WV-3089-CM','BT-7145-GW','LF-5723-ZQ','LN-1235-FF','BQ-2980-GR'],
    'Number_of_Views': [860.0, 3772.0, 3092.0, 466.0, 4426.0, 3444.0, 3171.0, 2919.0, 130.0, 1685.0, 769.0, 2391.0, 2433.0, 1184.0, 4555.0, 3385.0, 4117.0, 4843.0, 2904.0, 474.0, 1082.0],
    'Merchant Rating': [2.5, 4.8, 3.9, 3.4, 1.6, 1.6, 1.2, 4.5, 3.4, 3.8, 1.1, 4.9, 4.3, 1.8, 1.7, 1.7, 2.2, 3.1, 2.7, 2.2, 3.4]
}

# Convert dictionary into a pandas DataFrame
df = pd.DataFrame(data)

print("Dataset successfully loaded!")
print(df.head())
print("\nTotal number of rows:", len(df))

Dataset successfully loaded!
   product ID                                      Product Title  Merchant ID  \
0           1                    apple iphone 8 plus 64gb silver            1   
1           2                apple iphone 8 plus 64 gb spacegrau            2   
2           3  apple mq8n2b/a iphone 8 plus 64gb 5.5 12mp sim...            3   
3           4                apple iphone 8 plus 64gb space grey            4   
4           5  apple iphone 8 plus gold 5.5 64gb 4g unlocked ...            5   

  Category Label _Product Code  Number_of_Views  Merchant Rating  
0  Mobile Phones    QA-2276-XC            860.0              2.5  
1  Mobile Phones    KA-2501-QO           3772.0              4.8  
2  Mobile Phones    FP-8086-IE           3092.0              3.9  
3  Mobile Phones    YI-0086-US            466.0              3.4  
4  Mobile Phones    NZ-3586-WP           4426.0              1.6  

Total number of rows: 21


## 🧹 2. Data Cleaning & Preprocessing
- Checking for missing or null values in the dataset.
- Inspecting column data types and basic statistical summaries.

In [4]:
# Check for missing values in each column
print("Missing values per column:")
print(df.isnull().sum())

# Inspect column data types and general dataframe info
print("\nDataset Info:")
print(df.info())

# Display a quick preview of the columns
print("\nFirst 3 rows:")
print(df.head(3))

Missing values per column:
product ID         0
Product Title      0
Merchant ID        0
Category Label     0
_Product Code      0
Number_of_Views    0
Merchant Rating    0
dtype: int64

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21 entries, 0 to 20
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   product ID       21 non-null     int64  
 1   Product Title    21 non-null     object 
 2   Merchant ID      21 non-null     int64  
 3   Category Label   21 non-null     object 
 4   _Product Code    21 non-null     object 
 5   Number_of_Views  21 non-null     float64
 6   Merchant Rating  21 non-null     float64
dtypes: float64(2), int64(2), object(3)
memory usage: 1.3+ KB
None

First 3 rows:
   product ID                                      Product Title  Merchant ID  \
0           1                    apple iphone 8 plus 64gb silver            1   
1           2                apple iph

## ⚙️ 3. Feature Engineering
- Extracting supplementary features from product titles:
  - Title length (character count)
  - Word count
  - Presence of numbers (e.g., model numbers, storage sizes)

In [5]:
# Handle potential missing values in 'Product Title' before feature engineering
df['Product Title'] = df['Product Title'].fillna('')

# Create new numerical features
df['title_length'] = df['Product Title'].apply(len)
df['word_count'] = df['Product Title'].apply(lambda x: len(x.split()))
df['has_numbers'] = df['Product Title'].apply(lambda x: int(any(char.isdigit() for char in x)))

print("Feature engineering completed successfully!")
print("\nPreview of new features:")
print(df[['Product Title', 'title_length', 'word_count', 'has_numbers']].head())

Feature engineering completed successfully!

Preview of new features:
                                       Product Title  title_length  \
0                    apple iphone 8 plus 64gb silver            31   
1                apple iphone 8 plus 64 gb spacegrau            35   
2  apple mq8n2b/a iphone 8 plus 64gb 5.5 12mp sim...            70   
3                apple iphone 8 plus 64gb space grey            35   
4  apple iphone 8 plus gold 5.5 64gb 4g unlocked ...            54   

   word_count  has_numbers  
0           6            1  
1           7            1  
2          13            1  
3           7            1  
4          11            1  


## 🤖 4. Model Training & Comparison
- Splitting the dataset into training and testing sets.
- Converting text data (`Product Title`) into numerical vectors using TF-IDF (Term Frequency-Inverse Document Frequency).
- Training a baseline classification model and evaluating performance.

In [7]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# Check if we have at least 2 unique classes. If not, add a temporary mock row for demonstration.
if df['Category Label'].nunique() < 2:
    print("Note: The dataset currently contains only one category label. Adding a temporary mock category for pipeline demonstration.")
    dummy_row = pd.DataFrame({
        'product ID': [999],
        'Product Title': ['dell latitude laptop 15 inch i5 8gb'],
        'Merchant ID': [99],
        'Category Label': ['Laptops'],
        '_Product Code': ['DUMMY-01'],
        'Number_of_Views': [100.0],
        'Merchant Rating': [4.0],
        'title_length': [35],
        'word_count': [7],
        'has_numbers': [1]
    })
    df_model = pd.concat([df, dummy_row], ignore_index=True)
else:
    df_model = df.copy()

# Define features (X) and target (y)
X = df_model['Product Title']
y = df_model['Category Label']

# Split the dataset into training and testing sets safely
try:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
except ValueError:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Initialize TF-IDF Vectorizer for text feature extraction
vectorizer = TfidfVectorizer(stop_words='english')

# Transform training and testing text data
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

# Train the Logistic Regression classifier
model = LogisticRegression(random_state=42)
model.fit(X_train_vec, y_train)

# Make predictions on the test set
y_pred = model.predict(X_test_vec)

# Evaluate results
print("Model training and evaluation completed successfully!\n")
print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.4f}\n")
print("Classification Report:")
print(classification_report(y_test, y_pred, zero_division=0))

Note: The dataset currently contains only one category label. Adding a temporary mock category for pipeline demonstration.
Model training and evaluation completed successfully!

Accuracy Score: 1.0000

Classification Report:
               precision    recall  f1-score   support

Mobile Phones       1.00      1.00      1.00         5

     accuracy                           1.00         5
    macro avg       1.00      1.00      1.00         5
 weighted avg       1.00      1.00      1.00         5



## 💾 5. Saving the Final Model
- Serializing and saving the trained machine learning model and TF-IDF vectorizer to disk using Python's `pickle` library for future predictions.

In [8]:
import pickle

# Save the trained model to disk
model_filename = 'product_classifier_model.pkl'
with open(model_filename, 'wb') as file:
    pickle.dump(model, file)

# Save the fitted TF-IDF vectorizer to disk
vectorizer_filename = 'tfidf_vectorizer.pkl'
with open(vectorizer_filename, 'wb') as file:
    pickle.dump(vectorizer, file)

print(f"Model successfully saved as: {model_filename}")
print(f"Vectorizer successfully saved as: {vectorizer_filename}")

Model successfully saved as: product_classifier_model.pkl
Vectorizer successfully saved as: tfidf_vectorizer.pkl
